# Model Development
### DSCI 403
#### Clara Pugh and Chandler Bergstrom

This notebook loads in preprocessed data saved using preprocessing.ipynb, and begins to develop models to predict future data

*to see all datafiles and previous notebooks related to this project, visit the github: https://github.com/claranp/DSCI-403-Final-Project/settings

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, KFold

First, we will load in the data we saved from the preprocessing section:
- full_df will be the full clean dataframe
- surface_df will be the surface temperatures only, with one row per ARGO profile
- sst_df will be the raw satellite temp data, for use while plotting

In [2]:
# load in the preprocessed data that we saved
full_df = pd.read_csv('clean_data_gulfstream_jan_2023.csv')
full_df['TIME'] = pd.to_datetime(full_df['TIME'])

surface_df = pd.read_csv('clean_SURFACE_data_gulfstream_jan_2023.csv')
surface_df['TIME'] = pd.to_datetime(surface_df['TIME'])

# as well as the full SST dataframe
sst_df = pd.read_csv('SST_only_gulfstream_jan_2023_not_clean.csv')

full_df.head()

,index,N_PROF,N_LEVELS,CYCLE_NUMBER,DATA_MODE,DIRECTION,PLATFORM_NUMBER,PRES,PRES_ERROR,PSAL,PSAL_ERROR,TEMP,TEMP_ERROR,LATITUDE,LONGITUDE,TIME,matched_sst,match_dist_km,TEMP_MINUS_SST,MONTH_DAY
0,0,0,0,220,D,A,4903258,1.08,2.4,32.7404,0.01,6.448,0.002,42.82596,-52.65462,2023-01-01 02:16:37,9.15,48.544783,-2.702000,01-01
1,1,0,1,220,D,A,4903258,2.04,2.4,32.7411,0.01,6.454,0.002,42.82596,-52.65462,2023-01-01 02:16:37,9.15,48.544783,-2.696000,01-01
2,2,0,2,220,D,A,4903258,2.96,2.4,32.7431,0.01,6.470,0.002,42.82596,-52.65462,2023-01-01 02:16:37,9.15,48.544783,-2.680000,01-01
3,3,0,3,220,D,A,4903258,3.96,2.4,32.7452,0.01,6.458,0.002,42.82596,-52.65462,2023-01-01 02:16:37,9.15,48.544783,-2.691999,01-01
4,4,0,4,220,D,A,4903258,5.04,2.4,32.7467,0.01,6.459,0.002,42.82596,-52.65462,2023-01-01 02:16:37,9.15,48.544783,-2.691000,01-01


To make a dataframe that is set up to predict ARGO temperatures at depth, based of satellite SST measurements, I will make a new dataframe that has one row per ARGO profile, with a column for its temp at 0m, 50m, 100m, 500m, and 1000m (as close as possible to that given the depths that measurements were taken, linearly interpolated if no measurements were taken near those depths). I will also include the columns for the platform number and cycle number(to identify the float), as well as latitide, longitide, time, and matched SST (which is the satellite measurement). I will use those last four features to try to predict the temperatures at depth (doing each separately). Latitude, longitude, and matched SST are already numeric, but TIME is in numpy datetime format, so I want to make a new time variable that is hour of year (hour zero is midnight on jan 1) to try to account for seasonal/time of day variations

In [3]:
target_depths = [0, 50, 100, 500, 1000]   # meters (~ dbar)
tol = 10          # if a measurement is within this many dbar of the target, use it directly, otherwise interpolate
profile_keys = ['PLATFORM_NUMBER', 'CYCLE_NUMBER']

def temp_at_depth(pres, temp, target, tol):
    """Temp at target pressure: nearest measurement if close enough, else linear interpolation.
    Outside the profile's range, np.interp holds the nearest endpoint value (no NaN)."""
    j = np.abs(pres - target).argmin()
    if abs(pres[j] - target) <= tol:
        return temp[j]
    return np.interp(target, pres, temp)

rows = []
for (plat, cyc), g in full_df.groupby(profile_keys):
    g = g.dropna(subset=['PRES', 'TEMP']).sort_values('PRES')
    if g.empty:
        continue
    pres, temp = g['PRES'].to_numpy(), g['TEMP'].to_numpy()

    row = {
        'PLATFORM_NUMBER': plat,
        'CYCLE_NUMBER': cyc,
        'LATITUDE': g['LATITUDE'].iloc[0],
        'LONGITUDE': g['LONGITUDE'].iloc[0],
        'TIME': g['TIME'].iloc[0],
        'matched_sst': g['matched_sst'].iloc[0],
    }
    for d in target_depths:
        row[f'TEMP_{d}m'] = temp_at_depth(pres, temp, d, tol)
    rows.append(row)

profile_df = pd.DataFrame(rows)

# hour of year: hour 0 = midnight on Jan 1 of that profile's year
year_start = pd.to_datetime(profile_df['TIME'].dt.year.astype(str) + '-01-01')
profile_df['HOUR_OF_YEAR'] = (profile_df['TIME'] - year_start).dt.total_seconds() / 3600

profile_df.head()

,PLATFORM_NUMBER,CYCLE_NUMBER,LATITUDE,LONGITUDE,TIME,matched_sst,TEMP_0m,TEMP_50m,TEMP_100m,TEMP_500m,TEMP_1000m,HOUR_OF_YEAR
0,1902299,76,25.25809,-61.35474,2023-01-05 17:22:38,25.27,25.923,26.218,24.229,15.933,6.689,113.377222
1,1902324,41,25.17894,-61.97222,2023-01-29 07:57:12,25.02,24.908,24.797,23.835,15.250,6.841,679.953333
2,1902375,42,30.18262,-61.77085,2023-01-03 10:49:47,23.52,23.766,23.778,21.795,16.438,7.243,58.829722
3,1902375,43,30.19902,-61.61930,2023-01-13 08:09:49,23.33,22.759,21.894,21.905,16.590,7.214,296.163611
4,1902375,44,30.26663,-61.39466,2023-01-23 05:35:59,22.31,22.935,22.334,21.999,16.800,7.297,533.599722


## Prep data for applying the models
1. Drop columns that we will not use for analysis, make 5 separate datasets for the 5 separate target depths
2. Split into training and testing data
3. Standardize features

In [4]:
# create X using only the features we are going to use
X = profile_df[['LATITUDE', 'LONGITUDE', 'matched_sst', 'HOUR_OF_YEAR']]

# create a separate y for each depth (they are all different targets that we'll deal with separately)
y_0 = profile_df['TEMP_0m']
y_50 = profile_df['TEMP_50m']
y_100 = profile_df['TEMP_100m']
y_500 = profile_df['TEMP_500m']
y_1000 = profile_df['TEMP_1000m']

In [5]:
# split into training and testing (75% training, 25% testing)
X_0_train, X_0_test, y_0_train, y_0_test = train_test_split(X, y_0, test_size=0.25, random_state=42)
X_50_train, X_50_test, y_50_train, y_50_test = train_test_split(X, y_50, test_size=0.25, random_state=42)
X_100_train, X_100_test, y_100_train, y_100_test = train_test_split(X, y_100, test_size=0.25, random_state=42)
X_500_train, X_500_test, y_500_train, y_500_test = train_test_split(X, y_500, test_size=0.25, random_state=42)
X_1000_train, X_1000_test, y_1000_train, y_1000_test = train_test_split(X, y_1000, test_size=0.25, random_state=42)

In [6]:
# standardize features
scaler = StandardScaler()

X_0_train = scaler.fit_transform(X_0_train)
X_0_test = scaler.transform(X_0_test)

X_50_train = scaler.fit_transform(X_50_train)
X_50_test = scaler.transform(X_50_test)

X_100_train = scaler.fit_transform(X_100_train)
X_100_test = scaler.transform(X_100_test)

X_500_train = scaler.fit_transform(X_500_train)
X_500_test = scaler.transform(X_500_test)

X_1000_train = scaler.fit_transform(X_1000_train)
X_1000_test = scaler.transform(X_1000_test)

## Build Linear Regression Model
Some hyperparameters:
- fit_intercept (default is True): If true, it calculates and intercept, if false, it assumes intercept to be zero. I will keep this default
- copy_X (default is True): If true it creates a copy of X, if false it overwrites X. I will keep this True to not overwrite X.
- n_jobs (default is None): number of jobs to use for computation, use to speed up large problems. I will use default on this as well.


I will evaluate the models based off of root mean square error (RMSE), which gives the prediction error in the same units as the target variable (°C), and $R^2$, which explains how much variance in the data is explained by the model (1 is 100%, 0 is no better than random)

In [7]:
#Initialize a Linear Regression model object and train (or fit) the data
#Note: This one uses OLS ... vs SGDRegressor for Gradient Descent
model = LinearRegression()

model.fit(X_0_train, y_0_train)
y_0_hat = model.predict(X_0_test)

model.fit(X_50_train, y_50_train)
y_50_hat = model.predict(X_50_test)

model.fit(X_100_train, y_100_train)
y_100_hat = model.predict(X_100_test)

model.fit(X_500_train, y_500_train)
y_500_hat = model.predict(X_500_test)

model.fit(X_1000_train, y_1000_train)
y_1000_hat = model.predict(X_1000_test)

In [8]:
# Evaluate the models
mse_0 = mean_squared_error(y_0_test, y_0_hat)
r2_0 = r2_score(y_0_test, y_0_hat)   
print(f"Predicing 0m depth:")
print(f"Root Mean Squared Error: {np.sqrt(mse_0):.2f}")
print(f"R-squared: {r2_0:.2f}\n")

mse_50 = mean_squared_error(y_50_test, y_50_hat)
r2_50 = r2_score(y_50_test, y_50_hat)   
print(f"Predicing 50m depth:")
print(f"Root Mean Squared Error: {np.sqrt(mse_50):.2f}")
print(f"R-squared: {r2_50:.2f}\n")

mse_100 = mean_squared_error(y_100_test, y_100_hat)
r2_100 = r2_score(y_100_test, y_100_hat)   
print(f"Predicing 100m depth:")
print(f"Root Mean Squared Error: {np.sqrt(mse_100):.2f}")
print(f"R-squared: {r2_100:.2f}\n")

mse_500 = mean_squared_error(y_500_test, y_500_hat)
r2_500 = r2_score(y_500_test, y_500_hat)  
print(f"Predicing 500m depth:")
print(f"Root Mean Squared Error: {np.sqrt(mse_500):.2f}")
print(f"R-squared: {r2_500:.2f}\n")

mse_1000 = mean_squared_error(y_1000_test, y_1000_hat)
r2_1000 = r2_score(y_1000_test, y_1000_hat)  
print(f"Predicing 1000m depth:")
print(f"Root Mean Squared Error: {np.sqrt(mse_1000):.2f}")
print(f"R-squared: {r2_1000:.2f}")


Predicing 0m depth:
Root Mean Squared Error: 0.67
R-squared: 0.98

Predicing 50m depth:
Root Mean Squared Error: 1.02
R-squared: 0.95

Predicing 100m depth:
Root Mean Squared Error: 1.96
R-squared: 0.77

Predicing 500m depth:
Root Mean Squared Error: 3.41
R-squared: 0.48

Predicing 1000m depth:
Root Mean Squared Error: 1.25
R-squared: 0.47


As I predicted, the shallower depths have much better predictions than the lower depths. I will try regression with a couple other polynomial degrees to see if that makes it any better

In [9]:
targets = {0: y_0, 50: y_50, 100: y_100, 500: y_500, 1000: y_1000}

for depth, y in targets.items():
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    print(f"--- {depth} m ---")
    for degree in [1, 2, 3]:
        pipe = make_pipeline(
            StandardScaler(),
            PolynomialFeatures(degree, include_bias=False),
            LinearRegression())
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        print(f"degree {degree}: RMSE = {rmse:.2f} °C, R² = {r2_score(y_test, pred):.3f}")

--- 0 m ---
degree 1: RMSE = 0.67 °C, R² = 0.976
degree 2: RMSE = 0.66 °C, R² = 0.976
degree 3: RMSE = 0.72 °C, R² = 0.972
--- 50 m ---
degree 1: RMSE = 1.02 °C, R² = 0.946
degree 2: RMSE = 0.89 °C, R² = 0.958
degree 3: RMSE = 1.06 °C, R² = 0.941
--- 100 m ---
degree 1: RMSE = 1.96 °C, R² = 0.772
degree 2: RMSE = 1.74 °C, R² = 0.820
degree 3: RMSE = 2.00 °C, R² = 0.760
--- 500 m ---
degree 1: RMSE = 3.41 °C, R² = 0.479
degree 2: RMSE = 2.65 °C, R² = 0.684
degree 3: RMSE = 2.91 °C, R² = 0.621
--- 1000 m ---
degree 1: RMSE = 1.25 °C, R² = 0.472
degree 2: RMSE = 1.06 °C, R² = 0.618
degree 3: RMSE = 1.08 °C, R² = 0.603


Interestingly, a 2nd degree polynomial seems to fit best for each of the five cases, although the predictions still get significantly worse with depth still.

## Build Random Forest Regression Model
Some hyperparameters:
- n_estimators (default is 100): the number of trees in the forest. I will use the default for now
- max_depth (default is None): the maximum depth of each tree. If None, trees grow until the leaves are pure or hit min_samples_leaf. I will keep this at default and instead change min_samples_leaf
- min_samples_split (default is 2): the minimum number of samples a node needs in order to be split. I will keep the default.
- min_samples_leaf (default is 1): the minimum number of samples required in a leaf. I will use 3 so that the model is not able to overfit to each individual profile.
- max_features (default is 1.0 (all features)): the number of features considered at each split. I will keep this default as we only have 4 features
- criterion (default is 'squared_error'): the function used to measure split quality. I will keep this as default

In [10]:
targets = {0: y_0, 50: y_50, 100: y_100, 500: y_500, 1000: y_1000}

for depth, y in targets.items():
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=3, random_state=42)
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)

    print(f"{depth} m: RMSE = {np.sqrt(mean_squared_error(y_test, pred)):.2f} °C, "
          f"R² = {r2_score(y_test, pred):.3f}")

0 m: RMSE = 0.56 °C, R² = 0.983
50 m: RMSE = 0.72 °C, R² = 0.972
100 m: RMSE = 1.54 °C, R² = 0.858
500 m: RMSE = 1.73 °C, R² = 0.865
1000 m: RMSE = 1.00 °C, R² = 0.662


The random forest has generally better results than the regression, especially for the deeper measurements, although the predictions still get worse with depth.

# Tuning the Models
### Linear Regression

In [11]:
# Hyperparameter values to try (degree of the polynomial features)
param_grid_lr = {'polynomialfeatures__degree': [1, 2, 3, 4, 5]}

# apply cross-validation (KFold since the target is continuous)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def get_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {'MAE': mean_absolute_error(y_true, y_pred),
            'MSE': mse,
            'RMSE': np.sqrt(mse),
            'R2': r2_score(y_true, y_pred)}

targets = {0: y_0, 50: y_50, 100: y_100, 500: y_500, 1000: y_1000}

for depth, y in targets.items():
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # scaling inside the pipeline so it is re-fit on each CV training fold
    pipe = make_pipeline(StandardScaler(),
                         PolynomialFeatures(include_bias=False),
                         LinearRegression())

    grid_search = GridSearchCV(estimator=pipe, param_grid=param_grid_lr,
                               cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=0)

    # Fit on the training data only
    grid_search.fit(X_train, y_train)

    # best_estimator_ is already refit on the full training set with the best parameters
    best_lr = grid_search.best_estimator_
    y_pred_tuned = best_lr.predict(X_test)

    # Original model = degree 1 (plain linear regression), for comparison on the same test set
    original_lr = make_pipeline(StandardScaler(), PolynomialFeatures(1, include_bias=False), LinearRegression())
    original_lr.fit(X_train, y_train)
    y_pred_original = original_lr.predict(X_test)

    print(f"--- {depth} m ---")
    print("Best parameters:", grid_search.best_params_)
    print(f"Best cross-validated RMSE: {-grid_search.best_score_:.3f} °C")

    results = pd.DataFrame({'Original (degree 1)': get_metrics(y_test, y_pred_original),
                            'Tuned': get_metrics(y_test, y_pred_tuned)}).round(3)
    print(results, "\n")

--- 0 m ---
Best parameters: {'polynomialfeatures__degree': 1}
Best cross-validated RMSE: 1.083 °C
      Original (degree 1)  Tuned
MAE                 0.427  0.427
MSE                 0.450  0.450
RMSE                0.671  0.671
R2                  0.976  0.976 

--- 50 m ---
Best parameters: {'polynomialfeatures__degree': 3}
Best cross-validated RMSE: 1.117 °C
      Original (degree 1)  Tuned
MAE                 0.577  0.646
MSE                 1.033  1.122
RMSE                1.016  1.059
R2                  0.946  0.941 

--- 100 m ---
Best parameters: {'polynomialfeatures__degree': 2}
Best cross-validated RMSE: 1.485 °C
      Original (degree 1)  Tuned
MAE                 1.155  1.185
MSE                 3.824  3.018
RMSE                1.956  1.737
R2                  0.772  0.820 

--- 500 m ---
Best parameters: {'polynomialfeatures__degree': 3}
Best cross-validated RMSE: 2.107 °C
      Original (degree 1)  Tuned
MAE                 2.878  1.843
MSE                11.605  8.448

### Random Forest Regression

In [13]:
# Tuning the random forest regressor, separately for each depth
from sklearn.model_selection import GridSearchCV, KFold

# Hyperparameter values to try
param_grid = {'n_estimators': [100, 200, 300],
              'max_depth': [10, 20, None],
              'min_samples_split': [2, 5, 10],
              'min_samples_leaf': [1, 2, 3, 4, 5],
              'max_features': ['sqrt', None]}

# apply cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def get_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {'MAE': mean_absolute_error(y_true, y_pred),
            'MSE': mse,
            'RMSE': np.sqrt(mse),
            'R2': r2_score(y_true, y_pred)}

targets = {0: y_0, 50: y_50, 100: y_100, 500: y_500, 1000: y_1000}

for depth, y in targets.items():
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    grid_search = GridSearchCV(estimator=RandomForestRegressor(random_state=42), param_grid=param_grid,
                               cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=0)

    # Fit on the training data only
    grid_search.fit(X_train, y_train)

    # best_estimator_ is now refit on the full training set with the best parameters
    best_rf = grid_search.best_estimator_
    y_pred_tuned = best_rf.predict(X_test)

    # original model from the Build section, for comparison on the same test set
    original_rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=3, random_state=42)
    original_rf.fit(X_train, y_train)
    y_pred_original = original_rf.predict(X_test)

    print(f"--- {depth} m ---")
    print("Best parameters:\n", grid_search.best_params_)
    print(f"Best cross-validated RMSE: {-grid_search.best_score_:.3f} °C")

    results = pd.DataFrame({'Original': get_metrics(y_test, y_pred_original),
                            'Tuned': get_metrics(y_test, y_pred_tuned)}).round(3)
    print(results, "\n")

--- 0 m ---
Best parameters:
 {'max_depth': 20, 'max_features': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
Best cross-validated RMSE: 1.141 °C
      Original  Tuned
MAE      0.392  0.419
MSE      0.319  0.370
RMSE     0.564  0.608
R2       0.983  0.980 

--- 50 m ---
Best parameters:
 {'max_depth': 20, 'max_features': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}
Best cross-validated RMSE: 1.168 °C
      Original  Tuned
MAE      0.462  0.460
MSE      0.525  0.487
RMSE     0.725  0.698
R2       0.972  0.974 

--- 100 m ---
Best parameters:
 {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best cross-validated RMSE: 1.628 °C
      Original  Tuned
MAE      0.943  0.937
MSE      2.378  2.609
RMSE     1.542  1.615
R2       0.858  0.844 

--- 500 m ---
Best parameters:
 {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 